# 🎬 VIBE Render — Full Pipeline

> **OmniVoice TTS + HyperFrames = video.mp4**

1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all** (Ctrl+F9)
3. Tải video về

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ca-certificates fonts-liberation libasound2 libatk-bridge2.0-0 libatk1.0-0 libcairo2 libcups2 libdbus-1-3 libgbm1 libglib2.0-0 libgtk-3-0 libnspr4 libnss3 libpango-1.0-0 libx11-6 libxcomposite1 libxdamage1 libxext6 libxfixes3 libxrandr2 libxtst6 xdg-utils ffmpeg 2>&1 | tail -3
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash - 2>&1 | tail -2
!apt-get install -y nodejs 2>&1 | tail -2
!pip install -q omnivoice gradio numpy soundfile 2>&1 | tail -3
!node --version && !ffmpeg -version | head -1
print('All deps ready')

In [ ]:
import os, json, time, numpy as np, torch, soundfile as sf
from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

PROJ = '/content/vibe_project'
os.makedirs(PROJ, exist_ok=True)

for i in range(30):
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        break
    time.sleep(1)

# 9 scenes — tiếng Việt có dấu đầy đủ
SCENES = [
  "Xin chào các cậu vợ! Đây là giọng nói được tạo miễn phí do chính tay anh code.",
  "Anh sẵn sàng share cho mấy cậu vợ sử dụng không giới hạn, khỏi phải nom nớp lo vừa tạo được hai câu đã hiện cái thông báo hết credit.",
  "Anh biết cảm giác đang hứng làm video thì bị báo giới hạn nó cay cỡ nào, nên anh làm luôn một trang web cho dùng thoải mái.",
  "Chỉ cần nhập văn bản, chọn giọng đọc rồi bấm tạo, vài giây là có file mang về dùng video.",
  "Không cần nạp xu, không cần mở ví mỗi lần bấm nút tạo.",
  "Anh chỉ mong mấy cậu vợ dùng thấy ổn thì giới thiệu thêm bạn bè vào xài, vậy là anh có động lực tiếp tục cập nhật thêm nhiều giọng đọc mới.",
  "Nếu một ngày nào đó mấy cậu vợ thấy anh vẫn còn đăng video thì có nghĩa là server vẫn chưa sập và anh vẫn còn đủ tiền trả hóa đơn.",
  "Link anh để ngay dưới phần bình luận, cứ vào test thoải mái.",
  "Nếu dùng xong mà thấy ngon thì nhớ thả cho anh một follow, còn nếu không ngon thì... quay lại chửi anh, anh fix tiếp!",
]

print('Loading OmniVoice...')
DEVICE = get_best_device()
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True)
SR = model.sampling_rate
print(f'Model ready - SR: {SR}Hz')

GEN_CFG = OmniVoiceGenerationConfig(num_step=32, guidance_scale=1.8, denoise=True, preprocess_prompt=True, postprocess_output=True, position_temperature=5.0, class_temperature=0.2)

audios = []
for i, text in enumerate(SCENES):
    print(f'Scene {i+1}/{len(SCENES)}...')
    a = model.generate(text=text.strip(), language='vi', speed=0.95, generation_config=GEN_CFG)[0]
    audios.append(a)
    if i < len(SCENES) - 1:
        audios.append(np.zeros(int(SR * 0.5)))

full = np.concatenate(audios)
sf.write(f'{PROJ}/audio.mp3', full, SR)
print(f'Audio: {len(full)/SR:.1f}s saved')


In [ ]:
# Download template files from vibe-share repo
!wget -q https://raw.githubusercontent.com/doanquangkien/vibe-share/main/templates/64/gen.js -O /content/vibe_project/gen.js
!wget -q https://raw.githubusercontent.com/doanquangkien/vibe-share/main/templates/64/index.html -O /content/vibe_project/index.html
!wget -q https://raw.githubusercontent.com/doanquangkien/vibe-share/main/templates/64/config.json -O /content/vibe_project/config.json

# Run gen.js to create s1.html...s9.html
!node /content/vibe_project/gen.js /content/vibe_project 2>&1
!ls /content/vibe_project/s*.html | wc -l
print('Scenes generated')

In [ ]:
import subprocess, time, os

PROJ = '/content/vibe_project'
print('Rendering...')
start = time.time()
r = subprocess.run(['npx', '--yes', 'hyperframes@0.6.40', 'render', PROJ, '--output', f'{PROJ}/video.mp4'],
    capture_output=True, text=True, timeout=600, cwd=PROJ)
t = time.time() - start

out = f'{PROJ}/video.mp4'
if r.returncode == 0 and os.path.exists(out):
    size_mb = os.path.getsize(out) / (1024*1024)
    print(f'SUCCESS! video.mp4: {size_mb:.1f} MB in {t:.0f}s')
    from google.colab import files
    files.download(out)
else:
    print(f'FAILED (exit {r.returncode}, {t:.0f}s)')
    if r.stderr: print('STDERR:', r.stderr[-600:])